In [3]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [33]:
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(test.shape)
print(test.head())

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

validation = train.iloc[:100].copy()
print(validation.shape)
print(validation.head())

(500, 7)
   id                                             prompt  \
0   1  Pick the best possible answer: What is the rel...   
1   2  What is the estimated redshift of CEERS-93316,...   
2   3  Pick the best possible answer: What is the rea...   
3   4  What is the significance of the redshift-dista...   
4   5  What is the Landau-Lifshitz-Gilbert equation u...   

                                                   A  \
0  For every eigenstate of one Hamiltonian, its p...   
1  Approximately z = 6.0, corresponding to 1 bill...   
2  The sun appears yellowish due to a reflection ...   
3  Observations of the redshift-distance relation...   
4  The Landau-Lifshitz-Gilbert equation is a diff...   

                                                   B  \
0  For every eigenstate of one Hamiltonian, its p...   
1  Approximately z = 16.7, corresponding to 235.8...   
2  The longer wavelengths of light, such as red a...   
3  Observations of the redshift-distance relation...   
4  The Landau

In [5]:
DEBERTA_PATH = "microsoft/deberta-v3-small"
ROBERTA_PATH = "FacebookAI/roberta-base"

In [6]:
AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-small",
    num_labels=5,
    ignore_mismatched_sizes=True
)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias         

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

In [7]:
AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/roberta-base",
    num_labels=5,
    ignore_mismatched_sizes=True
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [8]:
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_PATH)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH)

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_PATH)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH)

deberta_model.to(device).eval()
roberta_model.to(device).eval()

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias         

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [9]:
id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E",
}

labels = ["A","B","C","D","E"]

In [10]:
@torch.no_grad()
def predict_probs(text,
                  tokenizer,
                  model,
                  max_length=512):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length,
    )

    inputs = {k:v.to(device) for k,v in inputs.items()}

    logits = model(**inputs).logits

    probs = F.softmax(logits, dim=-1)

    return probs.squeeze().cpu().numpy()

In [11]:
def top3_from_probs(probs):

    idx = np.argsort(probs)[::-1][:3]

    return " ".join(id2label[i] for i in idx)


def top1(probs):

    return id2label[np.argmax(probs)]


def confidence(probs):

    return probs.max()

# Question 1

In [38]:
row = test.iloc[25]

text = row["prompt"]

deberta_probs = predict_probs(
    text,
    deberta_tokenizer,
    deberta_model
)

print(deberta_probs)

[0.4346 0.5654]


In [39]:
best = np.argmax(deberta_probs)

print(
    id2label[best],
    deberta_probs[best]
)

B 0.5654


In [40]:
roberta_probs = predict_probs(
    text,
    roberta_tokenizer,
    roberta_model
)

print(roberta_probs)

[0.53940463 0.46059537]


In [41]:
print(deberta_model.config.num_labels)
print(roberta_model.config.num_labels)

2
2


In [42]:
best = np.argmax(deberta_probs)

print(
    f"{id2label[best]}, {deberta_probs[best]:.4f}"
)

B, 0.5654


# Question 2

In [15]:
avg_probs = (
    deberta_probs +
    roberta_probs
) / 2

print(avg_probs)

print(
    top1(avg_probs)
)

[0.48698747 0.5130125 ]
B


# Question 3

In [16]:
weighted_probs = (
    0.7 * deberta_probs +
    0.3 * roberta_probs
)

print(weighted_probs)

print(
    top1(weighted_probs)
)

[0.4660206 0.5341747]
B


# Question 4

In [17]:
top3 = top3_from_probs(weighted_probs)

print(top3)

B A


# Question 5

In [18]:
predictions = []

for _, row in tqdm(test.iterrows(), total=len(test)):

    text = row["prompt"]

    d = predict_probs(
        text,
        deberta_tokenizer,
        deberta_model
    )

    r = predict_probs(
        text,
        roberta_tokenizer,
        roberta_model
    )

    final = (
        0.7*d +
        0.3*r
    )

    predictions.append(
        top3_from_probs(final)
    )

  0%|          | 0/500 [00:00<?, ?it/s]

In [19]:
submission = pd.DataFrame({
    "id": test["id"],
    "prediction": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,id,prediction
0,1,B A
1,2,B A
2,3,B A
3,4,B A
4,5,B A


In [20]:
print(len(submission))

500


# Question 6

In [21]:
instruction = (
    "Answer the following multiple-choice question carefully:\n\n"
)

In [22]:
tta_changes = 0

for i in tqdm(range(50)):

    text = test.loc[i,"prompt"]

    original = predict_probs(
        text,
        deberta_tokenizer,
        deberta_model
    )

    augmented = predict_probs(
        instruction + text,
        deberta_tokenizer,
        deberta_model
    )

    avg = (
        original +
        augmented
    ) / 2

    if top1(original) != top1(avg):
        tta_changes += 1

print(tta_changes)

  0%|          | 0/50 [00:00<?, ?it/s]

4


# Question 7

In [23]:
different = 0

for i in tqdm(range(100)):

    text = test.loc[i,"prompt"]

    d = predict_probs(
        text,
        deberta_tokenizer,
        deberta_model
    )

    r = predict_probs(
        text,
        roberta_tokenizer,
        roberta_model
    )

    ensemble = (
        0.7*d +
        0.3*r
    )

    if top1(d) != top1(ensemble):
        different += 1

print(different)

  0%|          | 0/100 [00:00<?, ?it/s]

9


In [24]:
gain = 0

for i in tqdm(range(100)):

    text = test.loc[i,"prompt"]

    d = predict_probs(
        text,
        deberta_tokenizer,
        deberta_model
    )

    r = predict_probs(
        text,
        roberta_tokenizer,
        roberta_model
    )

    ensemble = (
        0.7*d +
        0.3*r
    )

    if confidence(ensemble) > confidence(d):
        gain += 1

print(gain)

  0%|          | 0/100 [00:00<?, ?it/s]

11


# Question 8

In [25]:
changes = 0

for i in tqdm(range(100)):

    text = test.loc[i,"prompt"]

    d = predict_probs(
        text,
        deberta_tokenizer,
        deberta_model
    )

    r = predict_probs(
        text,
        roberta_tokenizer,
        roberta_model
    )

    ensemble = (
        0.7*d +
        0.3*r
    )

    if top3_from_probs(d) != top3_from_probs(ensemble):
        changes += 1

print(changes)

  0%|          | 0/100 [00:00<?, ?it/s]

9


In [26]:
def apk(actual, predicted, k=3):

    predicted = predicted[:k]

    score = 0
    hits = 0

    for i,p in enumerate(predicted):

        if p == actual and p not in predicted[:i]:

            hits += 1
            score += hits / (i+1)

    return score

# Question 9

In [34]:
scores = []

for i in tqdm(range(100)):

    text = validation.loc[i,"prompt"]

    answer = validation.loc[i,"answer"]

    d = predict_probs(
        text,
        deberta_tokenizer,
        deberta_model
    )

    r = predict_probs(
        text,
        roberta_tokenizer,
        roberta_model
    )

    ensemble = (
        0.7*d +
        0.3*r
    )

    ranking = [
        id2label[j]
        for j in np.argsort(ensemble)[::-1][:3]
    ]

    scores.append(
        apk(answer, ranking)
    )

map3 = np.mean(scores)

print(round(map3,4))

  0%|          | 0/100 [00:00<?, ?it/s]

0.38


# Question 10

In [35]:
def apk(actual, predicted, k=3):

    predicted = predicted[:k]

    score = 0
    hits = 0

    for i,p in enumerate(predicted):

        if p == actual and p not in predicted[:i]:

            hits += 1
            score += hits / (i+1)

    return score

In [36]:
scores = []

for i in tqdm(range(100)):

    text = validation.loc[i,"prompt"]

    answer = validation.loc[i,"answer"]

    d = predict_probs(
        text,
        deberta_tokenizer,
        deberta_model
    )

    r = predict_probs(
        text,
        roberta_tokenizer,
        roberta_model
    )

    ensemble = (
        0.7*d +
        0.3*r
    )

    ranking = [
        id2label[j]
        for j in np.argsort(ensemble)[::-1][:3]
    ]

    scores.append(
        apk(answer, ranking)
    )

map3 = np.mean(scores)

print(round(map3,4))

  0%|          | 0/100 [00:00<?, ?it/s]

0.38
